In [9]:
# Libraries we are going to need.
import os
import requests
import re
from time import sleep
#from dotenv import load_dotenv
# pip install requests newspaper3k beautifulsoup4 AND

from newspaper import Article
from datetime import date, timedelta
from bs4 import BeautifulSoup
from collections import defaultdict
import csv
import pandas as pd
import gensim.models
from gensim.models import Word2Vec


import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.decomposition import PCA



In [10]:
# List of query words
COUNTRIES = [
    # Major powers / G20
    "united_states", "china", "russia", "india", "germany", "france",
    "united_kingdom", "japan", "italy", "canada", "brazil", "australia",
    "south_korea", "saudi_arabia", "mexico", "indonesia", "turkey",
    "argentina", "south_africa",
    

    # Europe (expanded)
    "ukraine", "poland", "netherlands", "belgium", "sweden", "norway",
    "denmark", "finland", "switzerland", "austria", "spain", "portugal",
    "greece", "hungary", "czech_republic", "romania", "bulgaria",
    "serbia", "ireland",

    # Middle East
    "israel", "iran", "iraq", "syria", "lebanon", "jordan",
    "united_arab_emirates", "qatar", "kuwait", "oman", "yemen",

    # Africa
    "nigeria", "ethiopia", "egypt", "kenya", "south_africa",
    "ghana", "morocco", "algeria", "tunisia", "sudan",
    "democratic_republic_of_congo", "uganda", "tanzania",

    # Asia
    "pakistan", "bangladesh", "vietnam", "thailand", "philippines",
    "malaysia", "singapore", "taiwan", "north_korea",

    # Americas
    "chile", "peru", "colombia", "venezuela", "cuba",
    "bolivia", "ecuador", "paraguay", "uruguay",

    # Special / geopolitical
    "palestine", "kosovo", "hong_kong"
]

LEADERS = [
    # Major powers
    "joe_biden", "donald_trump", "xi_jinping", "vladimir_putin",
    "narendra_modi", "emmanuel_macron", "olaf_scholz",
    "keir_starmer", "rishi_sunak", "justin_trudeau",

    # Europe
    "volodymyr_zelenskyy", "ursula_von_der_leyen",
    "giorgia_meloni", "mark_rutte",

    # Middle East
    "benjamin_netanyahu", "ali_khamenei", "mohammed_bin_salman",
    "bashar_al_assad",

    # Asia
    "kim_jong_un", "lai_ching_te",

    # Americas
    "lula_da_silva", "javier_milei", "andres_manuel_lopez_obrador",

    # Africa
    "bola_tinubu", "abiy_ahmed",

    # International org leaders
    "antonio_guterres", "klaus_schwab"
]

EVENTS = [
    # Wars / conflicts
    "ukraine_war", "russia_ukraine_war", "gaza_war",
    "israel_hamas_war", "syrian_civil_war",

    # Political processes
    "brexit", "election", "midterm_election", "general_election",
    "referendum",

    # Global crises
    "covid_19", "pandemic", "inflation", "recession",
    "energy_crisis", "climate_change",

    # Policy / governance
    "sanctions", "trade_war", "immigration", "border_policy",
    "foreign_policy", "defense_policy",

    # Tech / geopolitics
    "artificial_intelligence", "ai_regulation",
    "cybersecurity", "surveillance",

    # Economy
    "global_economy", "interest_rates", "central_bank",
    "supply_chain", "oil_prices"
]

INSTITUTIONS = [
    "nato", "united_nations", "eu", "european_union",
    "world_bank", "imf", "world_health_organization",
    "wto", "g7", "g20", "brics",
    "white_house", "kremlin", "downing_street",
    "pentagon", "european_commission"
]

QUERY_TERMS = COUNTRIES + LEADERS + EVENTS + INSTITUTIONS

In [11]:
df2023 = pd.read_csv('guardian_2023.csv')
df2024 = pd.read_csv('guardian_2024.csv')
df2025 = pd.read_csv('guardian_2025.csv')
df2026 = pd.read_csv('guardian_2026.csv')

df = pd.concat([df2023, df2024, df2025, df2026])

#do .lower() here in order to turn all of the big text of news articles into lower case,
#remove the .lower() from later time
df['Text'] = df['Text'].str.lower()
print(df.head())
#right here, turn "united states" into united_states

# Helper function that takes in replacements, a dictionary with key (original pattern) : value (new desired pattern) pairings and text, some string. It will replace all instances of the key in replacements to be the value string of the dictionary
# Returns the original string with all desired replacements
def multiple_replace(replacements, text):
    regex = re.compile(
        "(%s)" % "|".join(map(re.escape, replacements.keys()))
    )
    return regex.sub(lambda mo: replacements[mo.group()], text)

# Create phrase_dict, a dictionary with original, two words string as keys and underscored version as values
# Example: {"United States": "United_States"}
phrase_dict = {
    term.replace("_", " "): term
    for term in QUERY_TERMS
    if "_" in term
}

# Sort phrase_map to ensure substring of other words occur first (ie Israel before Israel Hamas War) to ensure multiple replace does not miss multi word substrings of other multiwords
phrase_dict = dict(
    sorted(phrase_dict.items(), key=lambda x: len(x[0]), reverse=True)
)

df["Text_clean"] = df["Text"].apply(
    lambda text: multiple_replace(phrase_dict, text)
)

                                               Title      Publication Date  \
0  Why has the Adani Group shed US$90bn in value ...  2023-02-01T22:53:33Z   
1  Costa Rican farmer handed 22 years for murder ...  2023-02-01T22:38:53Z   
2  Welby ‘would rather see C of E disestablished ...  2023-02-01T20:09:27Z   
3  Boris Johnson calls on US to give Ukraine figh...  2023-02-01T20:03:38Z   
4  Watchdog looks into £220,000 public funding fo...  2023-02-01T20:00:40Z   

                                                 URL  \
0  https://www.theguardian.com/business/2023/feb/...   
1  https://www.theguardian.com/world/2023/feb/01/...   
2  https://www.theguardian.com/world/2023/feb/01/...   
3  https://www.theguardian.com/world/2023/feb/01/...   
4  https://www.theguardian.com/politics/2023/feb/...   

                                                Text  
0  the sprawling empire of gautam adani, an india...  
1  a costa rican court has sentenced a man to 22 ...  
2  the archbishop of canterbu

In [ ]:
#this is the part that takes a really long time to run
#runtime around 10 minutes on m1 pro mbp

guardian_sentences = [doc.split() for doc in df['Text_clean']]
model = Word2Vec(
    sentences=guardian_sentences,
    vector_size=200,
    window=8,
    min_count=2,
    workers=8,
    sg=1,
    epochs=10,
    negative=10,
    sample=1e-4
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [ ]:
#visualize with a graph

# Reduce dimensionality of word vectors for visualization
filtered_terms = [word for word in QUERY_TERMS if word in model.wv]

word_vectors = model.wv[filtered_terms]  # Get the word vectors


pca = PCA(n_components=2)  # Initialize PCA
result = pca.fit_transform(word_vectors)  # Fit and transform the word vectors

# Plot the words in a 2D space
plt.figure(figsize=(20, 16))
plt.scatter(result[:, 0], result[:, 1])

# Annotate words in the plot
words = list(model.wv.index_to_key)

for i, word in enumerate(filtered_terms):
    plt.annotate(word, (result[i, 0], result[i, 1]))

plt.title("Word Embeddings Visualization")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.grid()
plt.show()